In [1]:
!pip install -q sentence-transformers

import json
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score
import zipfile

In [ ]:
BASE_PATH = "/kaggle/input/datasets/thaiphonghuan/dataset-structure/dataset_structuring"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH = os.path.join(BASE_PATH, "val")
TEST_PATH = os.path.join(BASE_PATH, "test")

CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities', 
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sbert_model = SentenceTransformer('lwolfrum2/careerbert-g').to(device)
#SentenceTransformer('all-mpnet-base-v2')
#SentenceTransformer('BAAI/bge-base-en-v1.5')
#SentenceTransformer('thenlper/gte-base')
#SentenceTransformer('jjzha/jobbert-base-cased')
#SentenceTransformer('thenlper/gte-large')
#SentenceTransformer('swaraj20/resume_gte_embedding')
#SentenceTransformer('lwolfrum2/careerbert-g')

# Trích xuất Đặc trưng và Masking matrix

In [ ]:
def process_view(view_data):
    """Xử lý missing_section thành special token"""
    if view_data is None: return "[EMPTY_SECTION]", 0
    
    if isinstance(view_data, list):
        clean_list = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean_list:
            return "[EMPTY_SECTION]", 0
        return ". ".join(clean_list), 1
    
    val = str(view_data).lower()
    if val == "missing_section" or val == "" or val == "none":
        return "[EMPTY_SECTION]", 0
    return str(view_data), 1

def prepare_data_with_mask(path):
    features = []
    masks = []
    reg_labels = []
    bin_labels = []
    
    matched_files = glob.glob(os.path.join(path, "matched/*.json"))
    mismatch_files = glob.glob(os.path.join(path, "mismatch/*.json"))
    configs = [(matched_files, 1), (mismatch_files, 0)]
    
    print(f"Đang xử lý dữ liệu (kèm Masking): {path}")
    for files, b_label in configs:
        for file_path in tqdm(files, desc=f"Folder {b_label}"):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            cv_struct = data['resume_structured']
            jd_struct = data['job_description_structured']
            
            # Xử lý text và lấy trạng thái tồn tại (mask)
            cv_data = [process_view(cv_struct.get(v)) for v in CV_VIEWS]
            jd_data = [process_view(jd_struct.get(v)) for v in JD_VIEWS]
            
            cv_texts, cv_exists = zip(*cv_data)
            jd_texts, jd_exists = zip(*jd_data)
            
            # Tạo ma trận mask m_ij (10x5)
            # m_ij = 1 nếu cả CV_i và JD_j đều có dữ liệu
            m_ij = np.outer(np.array(cv_exists), np.array(jd_exists))
            
            # Encoding
            with torch.no_grad():
                cv_emb = sbert_model.encode(list(cv_texts), convert_to_tensor=True)
                jd_emb = sbert_model.encode(list(jd_texts), convert_to_tensor=True)
            
            # Cross-view similarity S_ij
            s_ij = util.cos_sim(cv_emb, jd_emb).cpu().numpy()
            
            features.append(s_ij.flatten()) # 50 chiều
            masks.append(m_ij.flatten())    # 50 chiều
            reg_labels.append(data['target_label'] / 10.0)
            bin_labels.append(b_label)
            
    return np.array(features), np.array(masks), np.array(reg_labels), np.array(bin_labels)

# Thực thi trích xuất
X_train, M_train, y_train, b_train = prepare_data_with_mask(TRAIN_PATH)
X_val, M_val, y_val, b_val = prepare_data_with_mask(VAL_PATH)
X_test, M_test, y_test, b_test = prepare_data_with_mask(TEST_PATH)

# Weighted sum

In [ ]:
import torch.nn.functional as F

# 1. Dataset đồng bộ: Thực hiện Hadamard Product (S * M)
class MultiTaskDataset(Dataset):
    def __init__(self, X, M, y_reg, y_bin):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.M = torch.tensor(M, dtype=torch.float32)
        self.y_reg = torch.tensor(y_reg, dtype=torch.float32).view(-1, 1)
        self.y_bin = torch.tensor(y_bin, dtype=torch.float32).view(-1, 1)
        
    def __len__(self): return len(self.X)
    
    def __getitem__(self, idx):
        # THỰC HIỆN HADAMARD PRODUCT: S_ij * m_ij
        # Chỉ những ô có dữ liệu ở cả 2 bên mới được giữ lại giá trị similarity
        masked_feat = self.X[idx] * self.M[idx] 
        return masked_feat, self.y_reg[idx], self.y_bin[idx] # Trả về 50 chiều

# 2. KIẾN TRÚC WEIGHTED SUM (Linear Baseline)
class WeightedSumFusion(nn.Module):
    def __init__(self, input_dim=50): # Đổi đầu vào về 50
        super().__init__()
        # Phép tính: Score = Sigmoid(w1*s1 + w2*s2 + ... + w50*s50 + b)
        # Đây chính là định nghĩa toán học của Tổng có trọng số có thể học được
        self.weighted_layer = nn.Linear(input_dim, 1) 
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x lúc này đã là 50 đặc trưng similarity được lọc bởi mask
        out = self.weighted_layer(x)
        return self.sigmoid(out)

# Khởi tạo DataLoader với logic 50 chiều
train_loader = DataLoader(MultiTaskDataset(X_train, M_train, y_train, b_train), batch_size=32, shuffle=True)
val_loader = DataLoader(MultiTaskDataset(X_val, M_val, y_val, b_val), batch_size=32)
test_loader = DataLoader(MultiTaskDataset(X_test, M_test, y_test, b_test), batch_size=32)

# Khởi tạo model với input_dim=50
model = WeightedSumFusion(input_dim=50).to(device)

criterion_reg = nn.MSELoss()
criterion_bin = nn.BCELoss()

# Optimizer và Scheduler giữ nguyên để đảm bảo công bằng khi so sánh SOTA
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

print("⚠️ Đã chuẩn bị kiến trúc WEIGHTED SUM (50 features - Hadamard Masking).")

# Training

In [ ]:
epochs = 150
best_val_loss = float('inf')

print("Bắt đầu huấn luyện Weighted Sum...")
for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    
    for bx, by_reg, by_bin in train_loader:
        bx, by_reg, by_bin = bx.to(device), by_reg.to(device), by_bin.to(device)
        optimizer.zero_grad()
        pred_score = model(bx)
        
        l_reg = criterion_reg(pred_score, by_reg)
        l_bin = criterion_bin(pred_score, by_bin)
        loss = l_reg + 0.5 * l_bin 
        
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for bx, by_reg, by_bin in val_loader:
            bx, by_reg, by_bin = bx.to(device), by_reg.to(device), by_bin.to(device)
            out = model(bx)
            val_loss += (criterion_reg(out, by_reg) + 0.5 * criterion_bin(out, by_bin)).item()
    
    avg_val = val_loss / len(val_loader)
    scheduler.step(avg_val)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {total_train_loss/len(train_loader):.6f} | Val: {avg_val:.6f}")
    
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "weighted_sum_best_model.pth")

print("Huấn luyện Weighted Sum hoàn tất!")

# Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score
import zipfile

model.load_state_dict(torch.load("weighted_sum_best_model.pth"))
model.eval()

all_preds = []
all_true_reg = []
all_true_bin = []

print("Đang đánh giá trên tập dữ liệu Test...")
with torch.no_grad():
    for bx, by_reg, by_bin in test_loader:
        bx = bx.to(device)
        outputs = model(bx)
        
        all_preds.extend(outputs.cpu().numpy().flatten())
        all_true_reg.extend(by_reg.numpy().flatten())
        all_true_bin.extend(by_bin.numpy().flatten())

all_preds = np.array(all_preds)
all_true_reg = np.array(all_true_reg)
all_true_bin = np.array(all_true_bin)

# Chuyển đổi điểm dự đoán sang nhãn nhị phân dựa trên ngưỡng 0.5
threshold = 0.5
all_preds_bin = (all_preds >= threshold).astype(int)

# Metrics tổng hợp
mae = mean_absolute_error(all_true_reg, all_preds)
mse = mean_squared_error(all_true_reg, all_preds)
acc = accuracy_score(all_true_bin, all_preds_bin)
pre = precision_score(all_true_bin, all_preds_bin)
rec = recall_score(all_true_bin, all_preds_bin)
f1 = f1_score(all_true_bin, all_preds_bin)

print("\n" + "="*50)
print(f"{'Metric':<25} | {'Value':<15}")
print("-" * 43)
print(f"{'Regression MAE':<25} | {mae:.4f}")
print(f"{'Regression MSE':<25} | {mse:.4f}")
print(f"{'Classification Acc':<25} | {acc*100:.2f}%")
print(f"{'Precision (Match)':<25} | {pre:.4f}")
print(f"{'Recall (Match)':<25} | {rec:.4f}")
print(f"{'F1-Score':<25} | {f1:.4f}")
print("="*50)

print("\nĐối chiếu 10 mẫu ngẫu nhiên trên tập Test:")
print(f"{'Thực tế Folder':<15} | {'Điểm gốc':<10} | {'Điểm dự đoán':<12} | {'Kết luận'}")
print("-" * 65)
for i in range(min(10, len(all_preds))):
    folder_name = "Matched" if all_true_bin[i] == 1 else "Mismatch"
    pred_label = "Match" if all_preds[i] >= threshold else "Mismatch"
    print(f"{folder_name:<15} | {all_true_reg[i]:<10.2f} | {all_preds[i]:<12.2f} | {pred_label}")

# 6. Lưu Model
MODEL_NAME = "weighted_sum_best_model.pth"
torch.save(model.state_dict(), MODEL_NAME)

with zipfile.ZipFile("matching_system_datn.zip", "w") as z:
    z.write(MODEL_NAME)

print(f"\n✅ Đã lưu và đóng gói mô hình: /kaggle/working/matching_system_datn.zip")